# 02 - Data Cleaning

This notebook is used to clean and prepare the BoardGameGeek dataset for exploratory analysis and machine learning.

The goals of this notebook are:

- Load the raw dataset
- Create a separate working copy
- Handle missing values
- Correct unsuitable data types
- Investigate invalid and extreeme values
- Review repeated game names
- Save a cleaned dataset in the procesed data folder
- Document all cleaning decisions

## Load the Raw Dataset and Create a Working Copy

The raw dataset is loaded into `df_raw`.

A separate copy called `df_clean` is then created. All cleaning operations will be performed on `df_clean` so that the original raw dataset remains unchanged.

In [1]:
import pandas as pd
from pathlib import Path

# Find main project folder
current_folder = Path.cwd()
project_root = current_folder.parent

# Set the path to the original raw dataset
raw_data_path = project_root / "data" / "raw" / "bgg_dataset.csv"

# Load the untouched raw dataset
df_raw = pd.read_csv(raw_data_path, sep=";", decimal=",")

# Create seperate working copy for cleaning
df_clean = df_raw.copy()

print("Raw dataset shape:", df_raw.shape)
print("Working copy shape:", df_clean.shape)

Raw dataset shape: (20343, 14)
Working copy shape: (20343, 14)


In [2]:
# Confirm that both DataFrames currently contain the same data
print("Same data:", df_raw.equals(df_clean))

# Confirm that they are separate DataFrame objects in memorey
print("Same object:", df_raw is df_clean)

Same data: True
Same object: False


## Handle Missing Values

Before changing the dataset, the missing values in the working copy are reviewed again.

Different columns may require different solutions depending on their purpose, data type, and amount of missing information.

In [3]:
# Count missing values in the working copy
missing_counts = df_clean.isnull().sum()

# Show only colums containing missing values
missing_counts[missing_counts > 0]

ID                   16
Year Published        1
Owned Users          23
Mechanics          1598
Domains           10159
dtype: int64

In [4]:
# Inspect rows where ID or Year Published is missin
df_clean[
    df_clean["ID"].isnull()
    | df_clean["Year Published"].isnull()
][
    [
        "ID",
        "Name",
        "Year Published",
        "Owned Users",
        "Mechanics",
        "Domains"
    ]
]

,ID,Name,Year Published,Owned Users,Mechanics,Domains
10776,NaN,Ace of Aces: Jet Eagles,1990.0,NaN,NaN,NaN
10835,NaN,Die Erben von Hoax,1999.0,NaN,NaN,NaN
11152,NaN,Rommel in North Africa: The War in the Desert ...,1986.0,NaN,NaN,NaN
11669,NaN,Migration: A Story of Generations,2012.0,NaN,NaN,NaN
12649,NaN,Die Insel der steinernen Wachter,2009.0,NaN,NaN,NaN
12764,NaN,Dragon Ball Z TCG (2014 edition),2014.0,NaN,NaN,NaN
13282,NaN,Dwarfest,2014.0,NaN,NaN,NaN
13984,NaN,Hus,NaN,NaN,NaN,NaN
14053,NaN,Contrario 2,2006.0,NaN,NaN,NaN
14663,NaN,Warage: Extended Edition,2017.0,NaN,NaN,NaN


In [5]:
# Inspect the model-related values for rows with a missing ID
# in case they hold usefull info worth keeping
columns_to_review = [
    "Name",
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age",
    "Users Rated",
    "Rating Average",
    "Complexity Average"
]

df_clean[df_clean["ID"].isnull()][columns_to_review]

,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,Complexity Average
10776,Ace of Aces: Jet Eagles,1990.0,2,2,20,10,110,6.26,2.00
10835,Die Erben von Hoax,1999.0,3,8,45,12,137,6.05,2.00
11152,Rommel in North Africa: The War in the Desert ...,1986.0,2,2,0,12,53,6.76,4.00
11669,Migration: A Story of Generations,2012.0,2,4,30,12,49,7.20,2.00
12649,Die Insel der steinernen Wachter,2009.0,2,4,120,12,49,6.73,3.00
12764,Dragon Ball Z TCG (2014 edition),2014.0,2,2,20,8,33,7.03,2.50
13282,Dwarfest,2014.0,2,6,45,12,82,6.13,1.75
13984,Hus,NaN,2,2,40,0,38,6.28,2.00
14053,Contrario 2,2006.0,2,12,0,14,37,6.30,1.00
14663,Warage: Extended Edition,2017.0,2,6,90,10,49,7.64,3.00


In [6]:
# Inspect the full record where Year Published is missing
df_clean[df_clean["Year Published"].isnull()].T

,13984
ID,NaN
Name,Hus
Year Published,NaN
Min Players,2
Max Players,2
Play Time,40
Min Age,0
Users Rated,38
Rating Average,6.28
BGG Rank,13986


### Missing Values Cleaning Decisions

The rows with missing `ID` values still contain useful model-related information, such as player counts, play tine, rating average, and complexity average. Since `ID` is only an identifier and will not be used as a model feature, rows should not be removed only because `ID` is missing.

The single row with missing `Year Published` will be removed because `Year Published` is an initial candidate feature and only one row is affected.

Missing values in `Mechanics` and `Domains` will be replaced with `Unknown`. Thes are text-based feature columns, and removing rows with missing `Domains` would remove almost half of the dataset.

Missing values in `Owned Users` will not be filled at this stage because `Owned Users` is not part of the initial candidate feature set.

In [7]:
# Remove rows where Year Published is missing
df_clean = df_clean.dropna(subset=["Year Published"])

print("Shape after removing missing Year Published:")
print(df_clean.shape)

Shape after removing missing Year Published:
(20342, 14)


In [8]:
# Replace missing text values in "Mechanics" 
# and "Domains" with "Unknown"
df_clean["Mechanics"] = df_clean["Mechanics"].fillna("Unknown")
df_clean["Domains"] = df_clean["Domains"].fillna("Unknown")

# Check remaining missing values
df_clean.isnull().sum()

ID                    15
Name                   0
Year Published         0
Min Players            0
Max Players            0
Play Time              0
Min Age                0
Users Rated            0
Rating Average         0
BGG Rank               0
Complexity Average     0
Owned Users           22
Mechanics              0
Domains                0
dtype: int64

In [9]:
# Reset index after removing one row
# (when I removed one game, there was a "index gap" left)
df_clean = df_clean.reset_index(drop=True)

print("Shape after resetting index:")
print(df_clean.shape)

Shape after resetting index:
(20342, 14)


### Missing Values Cleaning Result

One row was removed because it had a missing `Year Published` value. This was done because `Year Published` is one of the initial candidate model features and only one row was affected.

Missing values in `Mechanics` and `Domains` were replecad with `Unknown`.

After these changes, the remaining missing values are only in `ID` and `Owned Users`. These columns are not part of the initial model feature set, so they do not need to be filled at this stage.

## Correct Column Data Types

Some columns were loaded with data types that do not fully match the values they represent.

The following changes will be considered:

- `Year Published` should be stored as a whole number.
- `ID` should be stored as a nullable whole number beacuse some values are missing.
- `Owned Users` should be stored as a nullable whole number because some values are missing.
- `Name`, `Mechanics`, and `Domains` should be stored as text.
- Rating and complexity columns should remain decimal numbers.

In [10]:
# Check whether if integer-like colums contains any decimal values
integer_like_columns = [
    "ID",
    "Year Published",
    "Owned Users"
]

for column in integer_like_columns:
    non_missing_values = df_clean[column].dropna()
    decimal_value_count = ((non_missing_values % 1) != 0).sum()

    print(f"{column}: {decimal_value_count} decimal values")

ID: 0 decimal values
Year Published: 0 decimal values
Owned Users: 0 decimal values


In [11]:
# Convert Year Published to standard integer type
# It no longer contains missing values
df_clean["Year Published"] = df_clean["Year Published"].astype("int64")

# Convert colums containing whole numbers and missing values
# to Pandas nullable integer type
# -> int64 normal integer type and cannot naturally hold missing value
# -> Int64 with captial I contains hole number nad <NA> missing value
df_clean["ID"] = df_clean["ID"].astype("Int64")
df_clean["Owned Users"] = df_clean["Owned Users"].astype("Int64")

# Standardase text-based columns
text_columns = [
    "Name",
    "Mechanics",
    "Domains"
]

for column in text_columns:
    df_clean[column] = df_clean[column].astype("string")

In [12]:
# Display the updated data types
df_clean.dtypes

ID                      Int64
Name                   string
Year Published          int64
Min Players             int64
Max Players             int64
Play Time               int64
Min Age                 int64
Users Rated             int64
Rating Average        float64
BGG Rank                int64
Complexity Average    float64
Owned Users             Int64
Mechanics              string
Domains                string
dtype: object

### Data Type Correction Result

The integer-like columns were checked before converson and did not contain genuine decimal values

The following data type changes were made:

- `Year Published` was converted to the standard `int64` integer type
- `ID` and `Owned Users` were converted to the nullable `Int64` integer type because they still contains missing values
- `Name`, `Mechanics`, and `Domains` were converted to the Pandas `string` type.
- Rating and complexity columns remain as decimal values using `float64`.

The updated data types now better represent the information stored in each column

## Investigate Invalid and Extreme Values

Some numeric columns contain values that may be invalid, unknown, or unusualy extreme

In this section, the values are investigated before any cleaning decisions are made

Areas to review include:

- unusualy old or non-positive publication years
- player counts of zero or very high player limits
- play times of zero or extremely long durations
- minimum age values of zero
- complexity values of zero
- relationships between minimum and maximum player counts

In [13]:
# Review the ranges and typical values of important numeric columns
columns_to_investigate = [
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age",
    "Users Rated",
    "Rating Average",
    "BGG Rank",
    "Complexity Average",
    "Owned Users"
]

# .T rotates the table so that each column appears
#  as a row which should make it me easier to read
df_clean[columns_to_investigate].describe().T

,count,mean,std,min,25%,50%,75%,max
Year Published,20342.0,1984.249877,214.003181,-3500.0,2001.0,2011.0,2016.0,2022.0
Min Players,20342.0,2.019713,0.690383,0.0,2.0,2.0,2.0,10.0
Max Players,20342.0,5.672402,15.231728,0.0,4.0,4.0,6.0,999.0
Play Time,20342.0,91.29707,545.460492,0.0,30.0,45.0,90.0,60000.0
Min Age,20342.0,9.601957,3.644926,0.0,8.0,10.0,12.0,25.0
Users Rated,20342.0,841.010864,3511.644023,30.0,55.0,120.0,385.0,102214.0
Rating Average,20342.0,6.403233,0.935933,1.05,5.82,6.43,7.03,9.58
BGG Rank,20342.0,10172.702979,5872.915096,1.0,5087.25,10172.5,15258.75,20344.0
Complexity Average,20342.0,1.991187,0.848924,0.0,1.33,1.97,2.54,5.0
Owned Users,20320.0,1408.457628,5040.179315,0.0,146.0,309.0,864.0,155312.0


In [14]:
# Count values that have suspicious values
# an may require further investigation
suspicious_value_counts = pd.Series({
    "Year Published <= 0":
        (df_clean["Year Published"] <= 0).sum(),

    "Min Players <= 0":
        (df_clean["Min Players"] <= 0).sum(),

    "Max Players <= 0":
        (df_clean["Max Players"] <= 0).sum(),

    "Max Players below Min Players":
        (df_clean["Max Players"] < df_clean["Min Players"]).sum(),

    "Play Time <= 0":
        (df_clean["Play Time"] <= 0).sum(),

    "Min Age <= 0":
        (df_clean["Min Age"] <= 0).sum(),

    "Complexity Average <= 0":
        (df_clean["Complexity Average"] <= 0).sum(),

    "Rating Average outside 0-10":
        (
            (df_clean["Rating Average"] < 0)
            | (df_clean["Rating Average"] > 10)
        ).sum(),

    "Owned Users below 0":
        (df_clean["Owned Users"] < 0).sum()
})

suspicious_value_counts

Year Published <= 0               195
Min Players <= 0                   46
Max Players <= 0                  161
Max Players below Min Players     125
Play Time <= 0                    556
Min Age <= 0                     1250
Complexity Average <= 0           426
Rating Average outside 0-10         0
Owned Users below 0                 0
dtype: int64

In [15]:
# Showes the oldest publication years in the dataset
df_clean[
    [
        "Name",
        "Year Published",
        "Rating Average",
        "Users Rated"
    ]
].sort_values("Year Published").head(20)

,Name,Year Published,Rating Average,Users Rated
8174,Senet,-3500,5.82,664
1275,Backgammon,-3000,6.54,11680
20218,Marbles,-3000,4.69,473
8924,The Royal Game of Ur,-2600,5.90,549
172,Go,-2200,7.64,14843
19647,Three Men's Morris,-1400,4.31,60
20001,Nine Men's Morris,-1400,5.36,1310
20341,Tic-Tac-Toe,-1300,2.68,3275
20340,Chutes and Ladders,-200,2.86,3783
15133,Petteia,-100,6.01,51


In [16]:
# Shows games with the highest maximum player counts
df_clean[
    [
        "Name",
        "Min Players",
        "Max Players",
        "Play Time",
        "Rating Average"
    ]
].sort_values("Max Players", ascending=False).head(20)

,Name,Min Players,Max Players,Play Time,Rating Average
8516,"I Don't Know, What Do You Want to Play?",2,999,5,6.76
7025,Start Player: A Kinda Collectible Card Game,2,999,1,6.49
10813,Scrimish Card Game,2,999,100,5.90
18780,The Hammer of Thor: The Game of Norse Mythology,1,362,120,4.95
4913,Linkee!,2,200,30,6.33
16538,Pit Fighter: Fantasy Arena,2,163,30,5.57
9983,Alchemidus,1,127,30,6.37
5820,Black Powder: Second Edition,2,120,360,7.35
8231,Ricochet,1,100,20,7.87
10130,Polyhedral Park Planner,1,100,30,7.29


In [17]:
# Shows games with the longest recorded play times.
df_clean[
    [
        "Name",
        "Year Published",
        "Min Players",
        "Max Players",
        "Play Time",
        "Rating Average"
    ]
].sort_values("Play Time", ascending=False).head(20)

,Name,Year Published,Min Players,Max Players,Play Time,Rating Average
13420,The Campaign for North Africa: The Desert War ...,1979,8,10,60000,6.10
3208,Case Blue,2007,2,2,22500,8.26
6035,1914: Offensive à outrance,2013,2,4,17280,7.98
7895,Atlantic Wall: D-Day to Falaise,2014,2,6,14400,8.08
1322,Empires in Arms,1983,2,7,12000,7.60
9497,Drang Nach Osten!,1973,2,4,12000,6.82
13607,The Enigma Box,2017,1,4,10000,6.92
6518,1985: Under an Iron Sky,2018,2,6,8640,9.12
5195,Atlanta Is Ours,2018,1,2,7920,8.71
3886,"The Greatest Day: Sword, Juno, and Gold Beaches",2015,2,8,6000,8.70


In [18]:
# Separate negative historical years from year-zero placeholders
year_value_breakdown = pd.Series({
    "Negative publication years":
        (df_clean["Year Published"] < 0).sum(),

    "Publication year equal to 0":
        (df_clean["Year Published"] == 0).sum(),

    "Positive publication years":
        (df_clean["Year Published"] > 0).sum()
})

year_value_breakdown

Negative publication years        10
Publication year equal to 0      185
Positive publication years     20147
dtype: int64

In [19]:
# Inspect examples where publication year is recorded as 0
df_clean.loc[
    df_clean["Year Published"] == 0,
    [
        "Name",
        "Year Published",
        "Users Rated",
        "Rating Average",
        "Complexity Average"
    ]
].head(20)

,Name,Year Published,Users Rated,Rating Average,Complexity Average
1043,Eat Poop You Cat,0,1589,7.45,1.11
1488,Carrom,0,1600,7.04,1.48
2839,Unpublished Prototype,0,804,6.90,2.47
2988,Traditional Card Games,0,885,6.61,1.95
2993,Riichi Mahjong,0,306,8.36,3.33
3587,Outside the Scope of BGG,0,580,6.68,1.70
3746,Mini Kubb,0,291,7.35,1.27
4090,Celebrities,0,271,7.21,1.19
4196,Dobble Free Demo Version,0,744,6.46,1.03
4694,Koi-Koi,0,373,6.67,1.62


In [20]:
# Check whether if maximum players is below minimum
# players when both values are positive
positive_player_order_issues = df_clean[
    (df_clean["Min Players"] > 0)
    & (df_clean["Max Players"] > 0)
    & (df_clean["Max Players"] < df_clean["Min Players"])
]

print(
    "Positive non-zero player-count contradictions:",
    len(positive_player_order_issues)
)

positive_player_order_issues[
    [
        "Name",
        "Min Players",
        "Max Players",
        "Year Published",
        "Rating Average"
    ]
].head(20)

Positive non-zero player-count contradictions: 0


,Name,Min Players,Max Players,Year Published,Rating Average


In [21]:
# Inspect examples where minimum or maximum players is recorded as 0
df_clean.loc[
    (df_clean["Min Players"] <= 0)
    | (df_clean["Max Players"] <= 0),
    [
        "Name",
        "Min Players",
        "Max Players",
        "Play Time",
        "Year Published"
    ]
].head(20)

,Name,Min Players,Max Players,Play Time,Year Published
2625,Decktet,0,0,30,2008
2839,Unpublished Prototype,0,0,0,0
2988,Traditional Card Games,0,0,0,0
3222,Stonewall Jackson's Way II: Battles of Bull Run,0,2,720,2013
3319,Journal 29: Interactive Book Game,1,0,120,2017
3587,Outside the Scope of BGG,0,0,0,0
3793,Kings of War,2,0,60,2012
4258,Old School Tactical: Volume 1 - Fighting on th...,2,0,60,2016
4413,The Big Taboo,4,0,60,2006
4503,"Warhammer 40,000: Assault On Black Reach",2,0,240,2008


In [22]:
# Inspect examples where Play Time is recorded as 0
df_clean.loc[
    df_clean["Play Time"] == 0,
    [
        "Name",
        "Year Published",
        "Min Players",
        "Max Players",
        "Play Time",
        "Rating Average"
    ]
].head(20)

,Name,Year Published,Min Players,Max Players,Play Time,Rating Average
2839,Unpublished Prototype,0,0,0,0,6.90
2988,Traditional Card Games,0,0,0,0,6.61
3587,Outside the Scope of BGG,0,0,0,0,6.68
4930,Miscellaneous Game Accessory,0,0,0,0,7.16
5178,Top Ten,2020,4,9,0,7.90
6466,Flesh and Blood,2019,2,2,0,8.68
7361,Star Fleet Battles Silver Anniversary Master R...,2004,0,0,0,8.59
7381,The Battles of Mollwitz 1741 and Chotusitz 1742,2017,2,2,0,8.51
7588,Footy Manager,2011,1,8,0,8.06
7608,Waterloo Campaign 1815,2019,2,2,0,7.34


In [23]:
# Inspect examples where Min Age is recorded as 0
df_clean.loc[
    df_clean["Min Age"] == 0,
    [
        "Name",
        "Year Published",
        "Min Age",
        "Users Rated",
        "Rating Average"
    ]
].head(20)

,Name,Year Published,Min Age,Users Rated,Rating Average
414,Fire in the Lake,2014,0,2258,8.07
614,Washington's War,2010,0,2375,7.63
1043,Eat Poop You Cat,0,0,1589,7.45
1353,RAF: The Battle of Britain 1940,2009,0,876,7.97
1367,Warfighter: The Tactical Special Forces Card Game,2014,0,1117,7.71
1523,Normandy '44,2010,0,796,7.89
1580,Time's Up! Edición Amarilla,2008,0,971,7.46
1581,Fighting Formations: Grossdeutschland Motorize...,2011,0,901,7.63
1641,Wings of War: Deluxe Set,2007,0,1132,7.26
1758,Holland '44: Operation Market-Garden,2017,0,569,8.35


In [24]:
# Inspect examples where Complexity Average is recorded as 0
df_clean.loc[
    df_clean["Complexity Average"] == 0,
    [
        "Name",
        "Year Published",
        "Users Rated",
        "Rating Average",
        "Complexity Average"
    ]
].head(20)

,Name,Year Published,Users Rated,Rating Average,Complexity Average
3897,Monikers: More Monikers,2018,166,8.23,0.0
5192,"Warhammer 40,000 (Ninth Edition)",2020,107,8.32,0.0
5667,Star Wars: Legion - Clone Wars Core Set,2019,101,8.39,0.0
6371,Exit: The Game - The Gate Between Worlds,2020,78,8.04,0.0
6455,Exceed: Street Fighter - M. Bison Box,2019,65,8.62,0.0
6522,Silver Coin,2020,87,7.90,0.0
6837,Middle-earth Strategy Battle Game: The Lord of...,2018,67,8.69,0.0
7178,Marvel Dice Masters: Iron Man and War Machine ...,2017,95,7.32,0.0
7333,Chimera & More,2017,75,7.50,0.0
7376,Omen: Fires in the East,2019,72,7.54,0.0


### Invalid and Extreme Value Decisions

Negative publication years are retained because they represent ancient games published before the common era.

A publication year of `0` is treated as an unknown year rather than a genuine historical year.

Zero values in `Min Players`, `Max Players`, `Play Time`, and `Min Age` are treated as missing information because these values do not represent realistic game characteristics.

A `Complexity Average` of `0` is also treated as missing because the BoardGameGeek complexity scale uses positive values and zero most likely means that no complexity rating was available.

Extremely high maximum player counts and very long play times are retained. Although unusual, the associated games suggest that these values may represent genuine mass-participation games or very long campaign games.

No rating values were outside the valid range of 0 to 10, and no negative ownership values were found.


In [25]:
# Columns where 0 represents unknown or unavailable information
zero_placeholder_integer_columns = [
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age"
]

# Replace zero placeholders with missing values
# Use Pandas nullable integers because these columns now contain missing data
for column in zero_placeholder_integer_columns:
    df_clean[column] = (
        df_clean[column]
        .mask(df_clean[column] == 0)
        .astype("Int64")
    )

# Complexity is a decimal measurement, so it remains float64
df_clean["Complexity Average"] = df_clean[
    "Complexity Average"
].mask(df_clean["Complexity Average"] == 0)

In [26]:
# Check the missing-value totals created by replacing invalid zero placeholders.
placeholder_columns = [
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age",
    "Complexity Average"
]

df_clean[placeholder_columns].isnull().sum()

Year Published         185
Min Players             46
Max Players            161
Play Time              556
Min Age               1250
Complexity Average     426
dtype: int64

In [27]:
# Confirm that no genuine player-count contradictions remain
# when both player-count values are available.
valid_player_counts = (
    df_clean["Min Players"].notna()
    & df_clean["Max Players"].notna()
)

remaining_player_contradictions = (
    df_clean.loc[valid_player_counts, "Max Players"]
    < df_clean.loc[valid_player_counts, "Min Players"]
).sum()

print(
    "Remaining player-count contradictions:",
    remaining_player_contradictions
)

# Display the updated data types.
df_clean[
    [
        "Year Published",
        "Min Players",
        "Max Players",
        "Play Time",
        "Min Age",
        "Complexity Average"
    ]
].dtypes

Remaining player-count contradictions: 0


Year Published          Int64
Min Players             Int64
Max Players             Int64
Play Time               Int64
Min Age                 Int64
Complexity Average    float64
dtype: object

### Invalid and Extreme Value Result

Negative publication years were retained because they represent ancient historical games.

Zero values in `Year Published`, `Min Players`, `Max Players`, `Play Time`, `Min Age`, and `Complexity Average` were converted to missing values because they represented unknown or unavailable information.

After replacing zero player-count placeholders, no genuine cases remained where `Max Players` was lower than `Min Players`.

Extremely high maximum player counts and and very long play times were retained because they may represent genuine mass-participation games or unusually long campaign games.

The affected whole-number columns now use the nullable `Int64` data type, while `Complexity Average` remains `float64`.

## Review Repeated Game Names

The dataset contains repeated game names, but repeated names do not automatically mean that the records are duplicates.

Games may share a title while representing different editions, publication years, versions, or unrelated games. These records are reviewed before deciding whether anything should be removed.

In [28]:
# Count fully duplicated rows and repeated game-name occurrences
fully_duplicated_rows = df_clean.duplicated().sum()
repeated_name_occurrences = df_clean["Name"].duplicated().sum()

# Count the number of distinct titles that appear more than once
repeated_name_mask = df_clean["Name"].duplicated(keep=False)
unique_repeated_names = df_clean.loc[
    repeated_name_mask,
    "Name"
].nunique()

print("Fully duplicated rows:", fully_duplicated_rows)
print("Repeated name occurrences:", repeated_name_occurrences)
print("Unique names appearing more than once:", unique_repeated_names)

Fully duplicated rows: 0
Repeated name occurrences: 367
Unique names appearing more than once: 323


In [29]:
# Display examples of repeated names and their game details
repeated_name_rows = (
    df_clean[df_clean["Name"].duplicated(keep=False)]
    .sort_values(["Name", "Year Published"])
)

repeated_name_rows[
    [
        "ID",
        "Name",
        "Year Published",
        "Min Players",
        "Max Players",
        "Play Time",
        "Rating Average",
        "Complexity Average"
    ]
].head(30)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Rating Average,Complexity Average
13930,17183,1001,1900,2,4,60,6.43,2.00
14683,205495,1001,2016,2,8,45,6.11,2.00
19214,4456,1862,1990,1,2,180,4.68,2.38
9693,11284,1862,2000,4,7,360,7.35,3.71
19337,8308,3D Labyrinth,2002,2,4,30,5.13,1.06
14750,274205,3D Labyrinth,2019,2,4,30,6.10,1.50
19973,16387,3D Tic Tac Toe,1953,2,2,1,4.22,1.25
18552,67364,3D Tic Tac Toe,<NA>,2,2,10,4.81,1.00
8318,73312,4 Seasons,2010,2,2,15,6.33,1.83
14290,213304,4 Seasons,2016,3,4,30,6.37,2.00


### Repeated Game Names Result

Cleanned dataset contains no fully duplicated rows.

There are 367 repeated occurrences in the `Name` column, representing 323 distinct game names that appear more than once.

The reviewed examples have different IDs, publication years, player counts, play times, ratings, or complexity values. These differences suggest that repeated names may represent seperate editions, versions, or unrelated games sharing the same title.

Therefore, no records will be removed based only on duplicated game names.

## Save the Cleaned Dataset

The cleaned working copy is saved in the `data/processed` folder.

The original raw dataset remais unchanged in `data/raw`, while the processed file contains the cleaning decisions applied in this notebook.

In [30]:
# Perform final checks before saving the cleaned dataset
final_validation = pd.Series({
    "Rows": df_clean.shape[0],
    "Columns": df_clean.shape[1],
    "Fully duplicated rows": df_clean.duplicated().sum(),
    "Missing target values": df_clean["Rating Average"].isnull().sum()
})

final_validation

Rows                     20342
Columns                     14
Fully duplicated rows        0
Missing target values        0
dtype: int64

In [31]:
# Set the path for the cleaned dataset
# data/processed/bgg_cleaned.csv
processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "bgg_cleaned.csv"
)

# Save tat cleaned dataset without adding the DataFrame
# index as an extra CSV column (creates it)
df_clean.to_csv(
    processed_data_path,
    index=False,
    encoding="utf-8"
)

print("Cleaned dataset saved to:")
print(processed_data_path)
print("File exists:", processed_data_path.exists())

Cleaned dataset saved to:
c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\data\processed\bgg_cleaned.csv
File exists: True


In [32]:
# Reload the saved file to confirm that it can be opened successfully
df_saved_check = pd.read_csv(processed_data_path)

print("Saved file shape:", df_saved_check.shape)

print(
    "Column names match:",
    df_saved_check.columns.tolist() == df_clean.columns.tolist()
)

Saved file shape: (20342, 14)
Column names match: True


## Data Cleaning Summary

The raw BoardGameGeek dataset originally contaned 20,343 rows and 14 columns.

A separate working copy named `df_clean` was created so that the all cleaning operations could be performed without modifying the original raw dataset.

### Missing values

The single row with the missing `Year Published` value was removed because publicition year is one of the initial candidate model features and only one record was affected.

Missing values in the text-based `Mechanics` and `Domains` columns were replaced with `Unknown`. Removing rows with missing dmoain information would have removed almost half of the dataset.

Missing values remain in `ID` and `Owned Users`. These values were not filled because:

* `ID` is only an identifier and will not be not used as a prediction feature.
* `Owned Users` describes popularity after release and is not part of the initial candidate feature set.

### Corrected data types

The data types were adjusted to better present the information stored in each column:

* `ID` and `Owned Users` were converted to nullable `Int64` values.
* `Year Published` was initialy converted to an integer and later to nullable `Int64` after unknown year values were identified.
* `Min Players`, `Max Players`, `Play Time`, and `Min Age` use nullable `Int64` because they now contain missing values.
* `Name`, `Mechanics`, and `Domains` were converted to the Pandas `string` data tzpe.
* `Rating Average` and `Complexity Average` remain decimal values.

### Invalid and placeholder values

Negative publication years were retained because then represent ancient games published before the common era.

Zero values were treated as unavailable or unknown informacion in the following columns:

* `Year Published`
* `Min Players`
* `Max Players`
* `Play Time`
* `Min Age`
* `Complexity Average`

These zero placeholders were converted to missing values.

After replacing zero player-count placeholders, no genine records remained where `Max Players` was lower than `Min Players`.

Extremely high maximum player counts and very long play times were retained because the associated games suggest that these values may represent genuine mass-participation games or unusually long campaign games.

No `Rating Average` values were outside the expected range of 0 to 10, and no negative `Owned Users` values were found.

### Duplicate records and repeated names

The dataset contains no fully duplicated rows.

There are 367 repeated occurrences in the `Name` column, representing 323 distinct names that appear more than once.

The reviewed records contain differences in IDs, publication years, player counts, play times, ratings, or complexity values. These repeted names may represent different editions, versions, or unrelated games sharing the same title.

Therefore, no records were removed based only on repeated game names.

### Cleaned dataset

The cleaned dataset was saved as:

`data/processed/bgg_cleaned.csv`

The final claned dataset contains:

* 20,342 rows
* 14 columns
* 0 fully duplicated rows
* 0 missing values in the prediction target, `Rating Average`

The original raw dataset remains unchanged in the `data/raw` fodler.
